# 29. Biomarker Scoring Test (Pipeline 11)

- Goal: check biomarker derivation and movement-quality score records.
- Docs: `docs_eng/pipeline/11_biomarker_scoring.md` / `docs/pipeline/11_biomarker_scoring.md`
- Inputs: Feature records, biomech proxy records, and baseline config.
- Outputs: In-memory dataframes/reports unless a cell explicitly saves under `data/processed/`.
- Checks: Biomarker records, score records, data-confidence separation, and interpretation-rule outputs.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import pandas as pd

from movement.annotation import apply_annotation, load_annotation_csv
from movement.biomech import extract_rep_biomech
from movement.biomarker import BiomarkerRecord, from_biomech_record, from_feature_record
from movement.biomarker.scoring import BiomarkerScoreRecord, derive_biomarkers
from movement.config import LANDMARKS, make_coordinate_columns, make_required_columns, make_visibility_columns
from movement.exercise_definition import load_exercise_definition
from movement.features import extract_rep_features
from movement.io import load_pose_csv
from movement.normalization import normalize_pose_by_hip_torso
from movement.pipeline import (
    AnnotationConfig, BiomechConfig, BiomarkerConfig, ExerciseDefinitionConfig,
    FeaturesConfig, RoleContextConfig, NormalizationConfig,
    PhaseSegmentationConfig, PipelineConfig, ValidationConfig, run_pipeline,
)
from movement.segmentation import segment_phases
from movement.validation import run_basic_validation

print('imports OK')

## Data Setup

Runs full pipeline ①–⑩ to produce feature + biomech records.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing pyproject.toml")
    PROJECT_ROOT = PROJECT_ROOT.parent

csv_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv"
ann_path = PROJECT_ROOT / "data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv"
def_dir  = PROJECT_ROOT / "data/definitions/exercises"

df_raw = load_pose_csv(csv_path)
run_basic_validation(
    df=df_raw,
    required_columns=make_required_columns(LANDMARKS),
    coordinate_columns=make_coordinate_columns(LANDMARKS),
    visibility_columns=make_visibility_columns(LANDMARKS),
)
ann_df       = load_annotation_csv(ann_path)
df_ann, _    = apply_annotation(df_raw, ann_df)
exercise_def = load_exercise_definition(exercise_id='squat', definitions_dir=def_dir)
df_norm, _   = normalize_pose_by_hip_torso(df=df_ann, landmarks=LANDMARKS)
df_seg, _    = segment_phases(df_norm, exercise_def, fps_default=30.0)

feat_records   = extract_rep_features(df_seg, exercise_def)
biomech_records = extract_rep_biomech(df_seg, exercise_def, use_visibility_weight=True)

print(f'feature records : {len(feat_records)}')
print(f'biomech records : {len(biomech_records)}')

## Direct derive_biomarkers() Test

In [ ]:
baseline_path = PROJECT_ROOT / 'data/reference/baseline_zscore.json'

biomarker_records, score_records = derive_biomarkers(
    feat_records=feat_records,
    biomech_records=biomech_records,
    exercise_definition=exercise_def,
    definition_version=exercise_def.version,
    baseline_path=baseline_path,
)

print(f'BiomarkerRecord count    : {len(biomarker_records)}')
print(f'BiomarkerScoreRecord count: {len(score_records)}')

## Check 1: BiomarkerRecord Fields and irovenance

In [ ]:
assert len(biomarker_records) > 0, 'no BiomarkerRecords produced'
for r in biomarker_records:
    assert isinstance(r, BiomarkerRecord)
    assert r.biomarker_id,            f'biomarker_id empty'
    assert r.exercise_id == 'squat',  f'wrong exercise_id'
    assert r.value is not None
    assert len(r.source_fields) > 0,  f'source_fields empty for {r.biomarker_id}'
print(f'PASS: all {len(biomarker_records)} BiomarkerRecord fields valid')
print('Sample records:')
for r in biomarker_records[:4]:
    print(f'  {r.biomarker_id:45s}  rep={r.rep_id}  value={r.value:.4f}  unit={r.unit}')

## Check 2: BiomarkerScoreRecord — One ier Rep, Score 0–100

In [ ]:
assert len(score_records) > 0, 'no BiomarkerScoreRecords produced'
rows = []
for s in score_records:
    assert isinstance(s, BiomarkerScoreRecord)
    assert 0.0 <= s.final_score <= 100.0, f'score out of range: {s.final_score} (rep {s.rep_id})'
    assert s.rep_id is not None
    row = {'rep_id': s.rep_id, 'final_score': s.final_score}
    row.update({f'{k}_score': v for k, v in s.domain_scores.items()})
    rows.append(row)
print('PASS: BiomarkerScoreRecord scores within 0-100')
display(pd.DataFrame(rows))


## Check 3: Domain Score Weights (spatial 40 %, temporal 30 %, control 20 %, biomech 10 %)

In [ ]:
rows = []
for s in score_records:
    weights = s.domain_weights
    score_sum = sum(s.domain_scores.get(domain, 0.0) * weight for domain, weight in weights.items())
    weight_sum = sum(weights.values()) or 1.0
    reconstructed = score_sum / weight_sum
    rows.append({
        'rep_id': s.rep_id,
        'final_score': s.final_score,
        'weighted_domain_score': round(reconstructed, 4),
        'abs_diff': round(abs(s.final_score - reconstructed), 4),
        'domain_weights': weights,
    })
display(pd.DataFrame(rows))
print('NOTE: final score is inspected against the active domain_weights from the score record.')


## Check 4: Dynamic Floor > 0

In [ ]:
rows = []
for s in score_records:
    rows.append({
        'rep_id': s.rep_id,
        'final_score': s.final_score,
        'floor_applied': s.floor_applied,
        'num_withheld_features': len(s.withheld_features),
    })
display(pd.DataFrame(rows))
print('PASS: floor/withheld-feature provenance is available for all score records')


## Check 5: as_dict() Serialization

In [ ]:
required_score_keys = [
    'rep_id', 'exercise_id', 'final_score', 'domain_scores', 'floor_applied',
    'deductions', 'withheld_features', 'domain_weights', 'score_bounds',
]
for s in score_records:
    d = s.as_dict()
    for k in required_score_keys:
        assert k in d, f'missing key {k} in BiomarkerScoreRecord.as_dict()'
print('PASS: as_dict() contains all required current-contract keys')
print(json.dumps(score_records[0].as_dict(), indent=2, default=str))


## Check 6: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation          = ValidationConfig(enabled=True)
cfg.annotation          = AnnotationConfig(enabled=True, path=ann_path)
cfg.exercise_definition = ExerciseDefinitionConfig(enabled=True,
                              definitions_dir=def_dir, exercise_id='squat')
cfg.normalization       = NormalizationConfig(enabled=True)
cfg.phase_segmentation  = PhaseSegmentationConfig(enabled=True)
cfg.features.role_context = RoleContextConfig(enabled=True)
cfg.features            = FeaturesConfig(enabled=True)
cfg.biomech             = BiomechConfig(enabled=True)
cfg.biomarker           = BiomarkerConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS, ann_df=ann_df)

assert 'biomarker'        in pipe_report
assert 'biomarker_scores' in pipe_report
n_scores = len(pipe_report['biomarker_scores'])
print(f'PASS: pipeline ⑪ biomarker_scores: {n_scores} reps')
for s in pipe_report['biomarker_scores']:
    print(f'  rep={s["rep_id"]}  final_score={s["final_score"]:.2f}')
print(f'steps executed: {list(pipe_report.keys())}')

## Check Summary

This notebook cell is a compact execution/QC checkpoint. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.
